# Project 5 (FinQA rerun): Hybrid Search + Reranking

BM25 + dense hybrid retrieval, RRF fusion, cross-encoder reranking — on your
real FinQA data, real `qwen2.5:7b-instruct`, real `BAAI/bge-small-en-v1.5`
embeddings, real Qdrant.

**Carrying forward from Project 4's finding**: your FinQA questions already
contain strong literal anchors (dates, states, company-specific terms) that
overlap verbatim with the gold chunk text — that's *why* HyDE hurt (it threw
those anchors away). BM25 is built to exploit exactly this kind of literal
overlap, so this notebook is a direct test of that finding: expect BM25 to
hold its own (maybe win) on `control`-tagged questions, and dense to hold
its advantage on `vocab`-tagged ones.

**Prerequisites**
```bash
ollama serve
pip install langchain-classic langchain-community rank_bm25 sentence-transformers
```
Run from the project root (same level as `src/`, `data/`).

**New files** under `src/hybrid_search/`: `fusion.py` (same RRF as Project 4),
`hybrid.py` (real `BM25Retriever` + `EnsembleRetriever`), `reranker.py` (real
`CrossEncoder`), `ndcg.py`.


In [10]:
import sys, os, time, json
from pathlib import Path
from collections import Counter

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import load_embedder, load_llm
from src.utils.io import load_jsonl, save_json
from src.eval.retrieval_harness import load_index_chunks
from src.eval.sampling import stratified_sample_by_op

from src.hybrid_search.hybrid import build_bm25_retriever, build_dense_retriever, build_hybrid_retriever, run_fn_from_retriever
from src.hybrid_search.reranker import build_reranker, rerank
from src.hybrid_search.ndcg import ndcg_at_k

print("imports OK")


imports OK


## 0. Setup — same index, same sample as Project 4

Same `chunks.jsonl`, same `stratified_sample_by_op(per_op_n=10, seed=42)` as
Project 4's notebook, so results are directly comparable question-for-question
— this is a fresh notebook/kernel, so the index and sample are rebuilt here
rather than assuming Project 4's notebook state is still in memory.


In [6]:
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

chunks = load_index_chunks("data/processed/chunks.jsonl")
eval_examples = load_jsonl("data/processed/eval_dataset.jsonl")
print(f"{len(chunks)} indexable chunks, {len(eval_examples)} eval questions")

embeddings = load_embedder()

docs = [Document(page_content=c["text"], metadata={"chunk_id": c["chunk_id"], "doc_id": c["doc_id"]})
        for c in chunks]

client = QdrantClient(":memory:")
dim = len(embeddings.embed_query("dimension probe"))
client.create_collection(collection_name="finqa_hybrid_search",
                          vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
vectorstore = QdrantVectorStore(client=client, collection_name="finqa_hybrid_search", embedding=embeddings)

t0 = time.time()
vectorstore.add_documents(docs, batch_size=256)
print(f"indexed {len(docs)} chunks in {time.time()-t0:.1f}s")

# Same failure-mode tagging as Project 4, so results line up 1:1
def word_overlap_ratio(a: str, b: str) -> float:
    a_words = {w.lower().strip('.,;:()%$') for w in a.split()}
    b_words = {w.lower().strip('.,;:()%$') for w in b.split()}
    return len(a_words & b_words) / len(a_words) if a_words else 0.0

def tag_failure_mode(ex: dict, gold_text_lookup: dict) -> str:
    gold_ids = ex["gold_chunk_ids"]
    if len(gold_ids) >= 2:
        return "multihop"
    gold_texts = [gold_text_lookup[g] for g in gold_ids if g in gold_text_lookup]
    if gold_texts and max(word_overlap_ratio(ex["question"], gt) for gt in gold_texts) < 0.15:
        return "vocab"
    return "control"

gold_text_lookup = {c["chunk_id"]: c["text"] for c in chunks}
sample = stratified_sample_by_op(eval_examples, per_op_n=10, seed=42)
sample = [ex for ex in sample if ex["gold_chunk_ids"]]
for ex in sample:
    ex["failure_mode"] = tag_failure_mode(ex, gold_text_lookup)

print(f"sample size: {len(sample)}")
print("failure-mode composition:", Counter(ex["failure_mode"] for ex in sample))


8532 indexable chunks, 883 eval questions


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

indexed 8532 chunks in 88.0s
sample size: 64
failure-mode composition: Counter({'control': 34, 'multihop': 24, 'vocab': 6})


## 1. Concept — hybrid search + reranking

Dense retrieval alone can miss exact term matches; BM25 alone misses
semantic paraphrase. Run both, fuse ranked lists with RRF (real
`EnsembleRetriever`, not hand-rolled), then rerank the fused top candidates
with a cross-encoder — a small model that scores `[query, doc]` pairs
*together* (far more accurate than the independent-embedding comparison
both BM25 and dense retrieval structurally are), too slow to run over the
whole corpus but fine over a small candidate set.

**Architecture:**
```
query ──┬── BM25Retriever.invoke() ──┐
        └── dense.invoke()   ────────┤
                                      ├── EnsembleRetriever (RRF) → top-10
                                      │
                                top-10 candidates → CrossEncoder.rerank() → top-5
```

**Metric**: NDCG@10 this time (rank-sensitive — a hit at rank 1 counts more
than the same hit at rank 5, unlike plain Recall@k).


In [7]:
def ndcg_evaluate(run_fn, sample, k=10):
    from collections import defaultdict
    per_mode = defaultdict(list)
    latencies = defaultdict(list)
    for ex in sample:
        t0 = time.time()
        retrieved = run_fn(ex["question"])
        latencies[ex["failure_mode"]].append(time.time() - t0)
        score = ndcg_at_k(retrieved, set(ex["gold_chunk_ids"]), k)
        per_mode[ex["failure_mode"]].append(score)

    summary = {}
    for mode, scores in per_mode.items():
        summary[mode] = {"n": len(scores), "ndcg": sum(scores) / len(scores),
                          "avg_latency_s": sum(latencies[mode]) / len(latencies[mode])}
    all_scores = [s for scores in per_mode.values() for s in scores]
    all_lat = [l for lats in latencies.values() for l in lats]
    summary["ALL"] = {"n": len(all_scores), "ndcg": sum(all_scores) / len(all_scores),
                       "avg_latency_s": sum(all_lat) / len(all_lat)}
    return summary

K = 10


## 2. Dense-only (baseline — same retrieval as Project 4's baseline, NDCG@10 instead of Recall@5)

In [8]:
dense_retriever = build_dense_retriever(vectorstore, k=K)
dense_run = run_fn_from_retriever(dense_retriever, k=K)
dense_summary = ndcg_evaluate(dense_run, sample, k=K)
print(json.dumps(dense_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "ndcg": 0.34489183267487006,
    "avg_latency_s": 0.03315908710161845
  },
  "control": {
    "n": 34,
    "ndcg": 0.569805094125124,
    "avg_latency_s": 0.03253365264219396
  },
  "vocab": {
    "n": 6,
    "ndcg": 0.16666666666666666,
    "avg_latency_s": 0.02903147538503011
  },
  "ALL": {
    "n": 64,
    "ndcg": 0.4476683935070484,
    "avg_latency_s": 0.032439861446619034
  }
}


## 3. BM25-only

Real `BM25Retriever` (langchain_community, wraps `rank_bm25`) — no dense
embeddings involved at all, pure lexical term-frequency scoring.

**Prediction to check**: should hold up well on `control` (literal-anchor-heavy
by definition) and struggle relatively more on `vocab` (low literal overlap
by definition — BM25 has nothing to score against there).


In [14]:
bm25_retriever = build_bm25_retriever(chunks, k=K)
bm25_run = run_fn_from_retriever(bm25_retriever, k=K)
bm25_summary = ndcg_evaluate(bm25_run, sample, k=K)
print(json.dumps(bm25_summary, indent=2))


{
  "multihop": {
    "n": 24,
    "ndcg": 0.09259680011029171,
    "avg_latency_s": 0.017070502042770386
  },
  "control": {
    "n": 34,
    "ndcg": 0.2282899855042034,
    "avg_latency_s": 0.014909225351670208
  },
  "vocab": {
    "n": 6,
    "ndcg": 0.0,
    "avg_latency_s": 0.013852079709370932
  },
  "ALL": {
    "n": 64,
    "ndcg": 0.15600285484046744,
    "avg_latency_s": 0.015620596706867218
  }
}


## 4. Hybrid (RRF)

Real `EnsembleRetriever`. **Recall the Nimbus lesson**: equal 50/50 weights
aren't automatically right — they can drag a strong retriever down toward a
weaker one instead of lifting the weak one up. Testing default 50/50 first,
then a BM25-favoring weight given the literal-anchor finding.


In [15]:
hybrid_retriever_5050 = build_hybrid_retriever(vectorstore, chunks, k=K, bm25_weight=0.5, dense_weight=0.5)
hybrid_run_5050 = run_fn_from_retriever(hybrid_retriever_5050, k=K)
hybrid_summary_5050 = ndcg_evaluate(hybrid_run_5050, sample, k=K)
print("=== Hybrid RRF, 50/50 weights ===")
print(json.dumps(hybrid_summary_5050, indent=2))

# Given FinQA's literal-anchor finding, also try BM25-favoring weights
hybrid_retriever_7030 = build_hybrid_retriever(vectorstore, chunks, k=K, bm25_weight=0.7, dense_weight=0.3)
hybrid_run_7030 = run_fn_from_retriever(hybrid_retriever_7030, k=K)
hybrid_summary_7030 = ndcg_evaluate(hybrid_run_7030, sample, k=K)
print("\n=== Hybrid RRF, 70/30 (BM25-favoring) weights ===")
print(json.dumps(hybrid_summary_7030, indent=2))


=== Hybrid RRF, 50/50 weights ===
{
  "multihop": {
    "n": 24,
    "ndcg": 0.2561295671900457,
    "avg_latency_s": 0.05203527212142944
  },
  "control": {
    "n": 34,
    "ndcg": 0.4273736177733387,
    "avg_latency_s": 0.048513244180118334
  },
  "vocab": {
    "n": 6,
    "ndcg": 0.08333333333333333,
    "avg_latency_s": 0.04967482884724935
  },
  "ALL": {
    "n": 64,
    "ndcg": 0.33090332213835333,
    "avg_latency_s": 0.049942903220653534
  }
}

=== Hybrid RRF, 70/30 (BM25-favoring) weights ===
{
  "multihop": {
    "n": 24,
    "ndcg": 0.14163976707955914,
    "avg_latency_s": 0.050484309593836464
  },
  "control": {
    "n": 34,
    "ndcg": 0.24836820184947966,
    "avg_latency_s": 0.04842480491189396
  },
  "vocab": {
    "n": 6,
    "ndcg": 0.0,
    "avg_latency_s": 0.04623103141784668
  },
  "ALL": {
    "n": 64,
    "ndcg": 0.18506051988737074,
    "avg_latency_s": 0.048991452902555466
  }
}


## 5. Hybrid + Cross-Encoder Reranking

Real `sentence-transformers` `CrossEncoder` (`cross-encoder/ms-marco-MiniLM-L-6-v2`
— general-purpose, no finance-specific reranker exists off the shelf so this
is the honest default rather than reaching for something branded
'financial' but unvalidated). Retrieve top-10 with whichever hybrid weighting
won above, rerank down to top-5 equivalent (still scored at k=10 for NDCG
consistency with the other runs — reranking only reorders, doesn't shrink
the candidate pool here).


In [16]:
reranker_model = build_reranker()  # downloads ~80MB on first run

# Use whichever hybrid weighting scored higher above -- adjust this line
# once you see the 50/50 vs 70/30 numbers.
best_hybrid_retriever = hybrid_retriever_7030   # <-- change to hybrid_retriever_5050 if that wins

def hybrid_rerank_run(question, k=K):
    candidate_ids = run_fn_from_retriever(best_hybrid_retriever, k=k)(question)
    candidates = [{"chunk_id": cid, "text": gold_text_lookup.get(cid, "")} for cid in candidate_ids]
    reranked_ids = rerank(question, candidates, reranker_model, top_k=k)
    return reranked_ids

hybrid_rerank_summary = ndcg_evaluate(hybrid_rerank_run, sample, k=K)
print(json.dumps(hybrid_rerank_summary, indent=2))


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

{
  "multihop": {
    "n": 24,
    "ndcg": 0.1865652853981721,
    "avg_latency_s": 0.21243802706400552
  },
  "control": {
    "n": 34,
    "ndcg": 0.32738028686974874,
    "avg_latency_s": 0.17377670372233672
  },
  "vocab": {
    "n": 6,
    "ndcg": 0.0,
    "avg_latency_s": 0.15226570765177408
  },
  "ALL": {
    "n": 64,
    "ndcg": 0.24388275942386856,
    "avg_latency_s": 0.18625804409384727
  }
}


## 6. Full comparison + save results

In [17]:
all_results = {
    "dense_only": dense_summary,
    "bm25_only": bm25_summary,
    "hybrid_rrf_50_50": hybrid_summary_5050,
    "hybrid_rrf_70_30": hybrid_summary_7030,
    "hybrid_rerank": hybrid_rerank_summary,
    "sample_size": len(sample),
    "sample_failure_mode_composition": dict(Counter(ex["failure_mode"] for ex in sample)),
}

save_json(all_results, "data/processed/eval_results_hybrid_search.json")
print(json.dumps(all_results, indent=2))


{
  "dense_only": {
    "multihop": {
      "n": 24,
      "ndcg": 0.34489183267487006,
      "avg_latency_s": 0.03315908710161845
    },
    "control": {
      "n": 34,
      "ndcg": 0.569805094125124,
      "avg_latency_s": 0.03253365264219396
    },
    "vocab": {
      "n": 6,
      "ndcg": 0.16666666666666666,
      "avg_latency_s": 0.02903147538503011
    },
    "ALL": {
      "n": 64,
      "ndcg": 0.4476683935070484,
      "avg_latency_s": 0.032439861446619034
    }
  },
  "bm25_only": {
    "multihop": {
      "n": 24,
      "ndcg": 0.09259680011029171,
      "avg_latency_s": 0.017070502042770386
    },
    "control": {
      "n": 34,
      "ndcg": 0.2282899855042034,
      "avg_latency_s": 0.014909225351670208
    },
    "vocab": {
      "n": 6,
      "ndcg": 0.0,
      "avg_latency_s": 0.013852079709370932
    },
    "ALL": {
      "n": 64,
      "ndcg": 0.15600285484046744,
      "avg_latency_s": 0.015620596706867218
    }
  },
  "hybrid_rrf_50_50": {
    "multihop": {
    